# Investigate using a pretrained model

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.base import clone
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, precision_recall_fscore_support, accuracy_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.dummy import DummyClassifier
import timm
import pacmap
import json

## Resnet model

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
SEED = 82

## Prep metadata and dataset

In [ ]:
metadata_full = pd.read_csv("./sample_images/metadata.csv", index_col=0)
cols_to_keep = ["image_id", "width", "height", "file_name", "common_name", "supercategory"]
metadata = metadata_full[cols_to_keep].copy()

data_dir = Path("./sample_images/images")

class ImageDataset(Dataset):
    def __init__(self, file_names, root, transform):
        self.file_names = list(file_names)
        self.root = Path(root)
        self.transform = transform

    def __len__(self):
        return len(self.file_names)

    def __getitem__(self, i):
        img = Image.open(self.root / self.file_names[i]).convert("RGB")
        return self.transform(img)

## Generate embeddings and save to npy file

In [ ]:
def embed(model_name, file_names, root, batch_size=64, num_workers=4):
    model = timm.create_model(model_name, pretrained=True, num_classes=0).to(device).eval()
    cfg = timm.data.resolve_model_data_config(model)
    tfm = timm.data.create_transform(**cfg, is_training=False)

    ds = ImageDataset(file_names, root, tfm)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)

    out = []
    with torch.inference_mode():
        for batch in loader:
            out.append(model(batch.to(device)).cpu().numpy())
    return np.concatenate(out).astype(np.float32)

models = {
    "resnet18":  "resnet18.tv_in1k",
    "resnet50":  "resnet50.a1_in1k",
    "bit50":     "resnetv2_50x1_bit.goog_in21k",
    "dinov2_b":  "vit_base_patch14_dinov2.lvd142m",
}

for key, name in models.items():
    emb = embed(name, metadata["file_name"], data_dir)
    np.save(f"./embeddings/{key}_embeddings.npy", emb)
    print(key, emb.shape)

## Load embeddings

In [ ]:
emb_dir = Path("./embeddings")
emb_files = sorted(emb_dir.glob("*_embeddings.npy"))

le = LabelEncoder()
y = le.fit_transform(metadata["supercategory"])
class_names = le.classes_

# Compute 2D PaCMAP projections
projections = {}
for f in emb_files:
    key = f.stem.replace("_embeddings", "")
    X = np.load(f)
    assert len(X) == len(y), f"{key}: {len(X)} embeddings vs {len(y)} labels"
    X = StandardScaler().fit_transform(X)
    reducer = pacmap.PaCMAP(n_components=2, n_neighbors=None, MN_ratio=0.5, FP_ratio=2.0, random_state=SEED)
    projections[key] = reducer.fit_transform(X, init="pca")

# Plot one panel per embedding model, colored by class
n = len(projections)
ncols = min(4,n)
nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5.5 * nrows), squeeze=False)

cmap = plt.get_cmap("tab20")
colors = [cmap(i) for i in range(len(class_names))]

for ax, (key, Z) in zip(axes.ravel(), projections.items()):
    for i, cls in enumerate(class_names):
        mask = y == i
        ax.scatter(Z[mask, 0], Z[mask, 1], s=8, alpha=0.7, color=colors[i], label=cls)
    ax.set_title(key)
    ax.set_xticks([]); ax.set_yticks([])
for ax in axes.ravel()[n:]:
    ax.axis("off")

handles, labels = axes.ravel()[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="center right", markerscale=3, frameon=False, title="supercategory")
fig.tight_layout(rect=[0, 0, 0.88, 1])
plt.show()

## Data splits

In [ ]:
EMB_KEY = "dinov2_b"

In [ ]:
metadata_full = pd.read_csv("./sample_images/metadata.csv", index_col=0)
cols_to_keep = ["image_id", "width", "height", "file_name", "common_name", "supercategory"]
metadata = metadata_full[cols_to_keep].copy()

X = np.load(f"./embeddings/{EMB_KEY}_embeddings.npy")
le = LabelEncoder()
y = le.fit_transform(metadata["supercategory"])
class_names = le.classes_
assert len(X) == len(y)

# 70 / 15 / 15, stratified
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.15 / 0.85, stratify=y_trainval, random_state=SEED
)
print(X_train.shape, X_val.shape, X_test.shape)

## Hyperparameter tuning

In [ ]:
N_PCA = 15

def make_pipe(clf, pca=False):
    steps = [("scaler", StandardScaler())]
    if pca:
        steps.append(("pca", PCA(n_components=N_PCA, random_state=SEED)))
    steps.append(("clf", clf))
    return Pipeline(steps)

def to_jsonable(v):
    if isinstance(v, np.generic):
        return v.item()
    if isinstance(v, tuple):
        return list(v)
    return v

def from_jsonable(params):
    out = dict(params)
    if "clf__hidden_layer_sizes" in out:
        out["clf__hidden_layer_sizes"] = tuple(out["clf__hidden_layer_sizes"])
    return out

rf_grid  = {"clf__n_estimators": [200, 500], "clf__max_depth": [None, 10, 20]}
svm_grid = {"clf__C": [0.1, 1, 10, 100], "clf__gamma": np.logspace(-4, -1, 4)}
lr_grid  = {"clf__C": np.logspace(-4, 2, 7)}
mlp_grid = {
    "clf__hidden_layer_sizes": [(64,), (128,), (256,), (512,), (1024,), (128, 64), (256, 128), (1024, 512)],
    "clf__alpha": [1e-4, 1e-1, 0.3, 1.0, 10, 30, 100, 300, 1000],
}

experiments = {
    "RF (full)":   (make_pipe(RandomForestClassifier(random_state=SEED, n_jobs=-1)), rf_grid),
    "RF (PCA)":    (make_pipe(RandomForestClassifier(random_state=SEED, n_jobs=-1), pca=True), rf_grid),
    "SVM (full)":  (make_pipe(SVC(kernel="rbf")), svm_grid),
    "SVM (PCA)":   (make_pipe(SVC(kernel="rbf"), pca=True), svm_grid),
    "LogReg":      (make_pipe(LogisticRegression(max_iter=2000, random_state=SEED)), lr_grid),
    "MLP":         (make_pipe(MLPClassifier(max_iter=500, early_stopping=True, random_state=SEED)), mlp_grid),
}

In [ ]:
def score(model, X_, y_):
    pred = model.predict(X_)
    p, r, f, _ = precision_recall_fscore_support(y_, pred, average="macro", zero_division=0)
    return {"precision": p, "recall": r, "f1": f, "accuracy": accuracy_score(y_, pred)}, pred

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
searches, val_rows = {}, []

for name, (pipe, grid) in experiments.items():
    gs = GridSearchCV(pipe, grid, scoring="f1_macro", cv=cv, n_jobs=-1, refit=True)
    gs.fit(X_train, y_train)
    searches[name] = gs
    m, _ = score(gs.best_estimator_, X_val, y_val)
    val_rows.append({"model": name, "cv_f1": gs.best_score_, **{f"val_{k}": v for k, v in m.items()},
                     "best_params": {k.replace("clf__", ""): v for k, v in gs.best_params_.items()}})
    print(f"{name}: cv_f1={gs.best_score_:.3f}  val_f1={m['f1']:.3f}  {gs.best_params_}")

print("--------------------------")

for name, gs in searches.items():
    r = pd.DataFrame(gs.cv_results_).iloc[gs.best_index_]
    print(f"{name}: {r.mean_test_score:.3f} ± {r.std_test_score:.3f}")

# Dummy baseline
dummy = DummyClassifier(strategy="stratified", random_state=SEED).fit(X_train, y_train)
m, _ = score(dummy, X_val, y_val)
val_rows.append({"model": "Dummy", "cv_f1": np.nan, **{f"val_{k}": v for k, v in m.items()}, "best_params": {}})

val_results = pd.DataFrame(val_rows).set_index("model")
val_results.drop(columns="best_params").round(3)

best_params = {
    name: {
        "params": {k: to_jsonable(v) for k, v in gs.best_params_.items()},
        "cv_f1": float(gs.best_score_),
    }
    for name, gs in searches.items()
}

Path("./results").mkdir(exist_ok=True)
with open(f"./results/best_params_{EMB_KEY}.json", "w") as f:
    json.dump(best_params, f, indent=2)

val_results

## Training and performance

In [ ]:
test_rows, test_preds = [], {}


with open(f"./results/best_params_{EMB_KEY}.json") as f:
    best_params = json.load(f)

for name, (pipe, _) in experiments.items():
    params = from_jsonable(best_params[name]["params"])
    final = clone(pipe).set_params(**params).fit(X_trainval, y_trainval)
    m, pred = score(final, X_test, y_test)
    test_rows.append({"model": name, **m})
    test_preds[name] = pred
    print(name, {k: round(v, 3) for k, v in m.items()})

dummy_final = DummyClassifier(strategy="stratified", random_state=SEED).fit(X_trainval, y_trainval)
m, pred = score(dummy_final, X_test, y_test)
test_rows.append({"model": "Dummy", **m})
test_preds["Dummy"] = pred

test_results = pd.DataFrame(test_rows).set_index("model")
display(test_results.round(3))

# Confusion matrices
n = len(test_preds)
ncols = 3
nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(7 * ncols, 6 * nrows))
for ax, (name, pred) in zip(axes.ravel(), test_preds.items()):
    cm = confusion_matrix(y_test, pred)
    ConfusionMatrixDisplay(cm, display_labels=class_names).plot(
        ax=ax, xticks_rotation=45, colorbar=False, cmap="Blues"
    )
    ax.set_title(name)
for ax in axes.ravel()[n:]:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# classification_report(y_test, test_preds["SVM (full)"], target_names=class_names)